# Set Physics and Numerics

Nimbus includes many physical processes and numerical settings. Here we explain how these processes can be altered to improve your simulation. To start, we use the basic example setup.

In [1]:
import numpy as np
from nimbus import Nimbus

planet_gravity = 3100  # in cm/s^2
planet_mmw = 2.34  # mean molecular weight of the planet in amu
cloud_species = ['SiO2', 'MgSiO3']  # name of the cloud species
cloud_species_mmr = [3e-3, 5e-4]  # in [g/g] and same order as cloud_species
cloud_species_cant_nucleate = ['MgSiO3']  # species which cannot nucleate

# ==== read in the tp profile
tpname = 't1500g316f2_m0.0_co1.0.pt'
tp_profile = np.genfromtxt(tpname, skip_header=2)

# ==== prepare data for Nimbus
pressures = tp_profile[:, 1]  # in bars
temperatures = tp_profile[:, 2]  # in K
kzz = 1e10 * np.ones_like(temperatures)  # in cm2/s

# ==== Set up Nimbus object
nimb = Nimbus(working_dir='working/', verbose=True)

# ==== Set up Nimbus atmosphere structure
nimb.set_up_atmosphere(
    temperatures, pressures, kzz, planet_mmw,
    planet_gravity, cloud_species, cloud_species_mmr,
    ignore_as_nucleator=cloud_species_cant_nucleate
)

                   Welcome to Nimbus                       
[INFO] For questions contact: kiefersv.mail@gmail.com
[INFO] Settings selected:
       -> working directory: working/
       -> verbose: True
       -> analytic plots: False
[INFO] Atmosphere set up with:
       -> pressure range: 6.37e+01 - 1.78e-04 bar
       -> temperature range: 3.77e+03 - 7.52e+02 K
       -> Kzz range at t=0: 1.00e+10 - 1.00e+10 cm2/s
       -> Mean molecular weight: 2.34e+00 amu
       -> Gravity: 3.10e+03 cm/s2
       -> SiO2 deep MMR: 3.00e-03 g/g
       -> MgSiO3 deep MMR: 5.00e-04 g/g


## Explore the Microphysics
Nimbus includes a range of microphysical processes. In many cases, the thermodynamic data needed to model these processes are uncertain. To explore the effect of uncertainties, Nimbus allows to fudge the rates of these processes. From the basic setup, the physical parameters can be adjusted.

In [6]:
# Halving the efficiency of the ...
nimb.nucleation_efficiency = 0.5  # nucleation rate
nimb.growth_efficiency = 0.5  # growth rate
nimb.coal_efficiency = 0.5  # gravitational coalescence efficiency
nimb.coag_efficiency = 0.5  # coagulation efficiency

# The sticking coefficient affects the growth rate in collisional dominated regimes
nimb.sticking_coefficient = 0.5  # here it is set to 0.5

## Adjust the Numerics

Nimbus uses the solve_ivp function of scipy to solve the master equation. The standard inputs are listed below, and all can be changed. Some general recommendations are listed here as well.

In [8]:
# ==== type of solver from solve_ivp
# Recommended are 'LSODA' and 'BDF'. For more options see
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html
nimb.solver_type = 'LSODA'

# ==== solver tolerances
# Relative error should be below 1e-1 and absolute tolarance should be below 1e-10.
# In some cases setting ode_rtol to 1e-6 can help to resolve stuck cloud profile.
nimb.ode_rtol = 1e-3  # relative error of solve_ivp
nimb.ode_atol = 1e-25  # absolute error of solve_ivp

# ==== Minimum values
# These values prevent underflow errors. Typically, these values should be
# between 1e-20 and 1e-50.
nimb.ode_minimum_mmr = 1e-30  # lowest MMR considered [g/g]
nimb.minimum_nuc_rate = 1e-20  # lowest J value [1/cm3/s]

# ==== time steps
# The start time should be below 1 second. The end time can be whatever you need,
# typically cloud structures converge after 1e8. The time stepping is for output
# use only and can be set to any value above 2.
nimb.tstart = 1e-4  # start time of simulation [s]
nimb.tend = 1e12  # end time of simulation [s]
nimb.tsteps = 200  # number of intermediate evaluations (log-spaced)